# AI Marketplaces and Aggregation Rules — A Hands-On Tutorial

**Course context.** Week 5 lecture (`docs/05.md` §2): Donahue & Raghavan show that scoring rules over AI *producers* shape what gets built. Winrate-based leaderboards push producers toward homogenisation; weighted variants reward specialisation.

**What this notebook does.** Build a toy AI marketplace with **2 producers** competing for **200 consumers** spread across a 1-D taste axis. Run best-response dynamics under two scoring rules — *winrate* and *weighted winrate* — and watch the producer ecosystem either collapse onto one cluster or split to cover both.

**Prerequisites.** `numpy`, `matplotlib`. No API key, no internet. Runs in under a minute.

```bash
pip install numpy matplotlib
```


## §1 — From rules over outputs to rules over producers

In the Week 4 tutorial (`04_preference_aggregation/`), the alternatives were *given* — three LLM responses — and the aggregation rule chose a winner. Classical social choice.

In a marketplace, **the alternatives are produced by strategic actors who respond to whatever rule ranks them.** The same scoring function does double duty: it ranks model outputs *and* it shapes which models get built. This is the setting Donahue & Raghavan analyse.

**The question.** Holding consumers fixed, does the choice of leaderboard scoring rule determine whether the model ecosystem looks like a monoculture or a healthy variety?


## §2 — A toy AI marketplace

- **200 consumers** on a 1-D taste axis $[0, 1]$. Imagine the axis as a single value dimension — say *warmth* (left) vs *clinical formality* (right). The consumer distribution is **bimodal**: 100 at $\approx 0.3$, 100 at $\approx 0.7$. (One way to think about it: half the audience wants empathy from an AI, half wants efficiency.)
- **2 producers**, each picks a position in $[0, 1]$ — their model's behaviour on the taste axis.
- **Utility**: a consumer's utility from a producer is $1 - |p - c|$ (closer = better, max 1).
- **Score**: depends on the aggregation rule (next section).

Producers are strategic — they pick positions to maximise their score. Consumers are passive in this toy: each consumer is "won" by whichever producer is closest.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Bimodal consumer distribution: two clusters of equal size
N = 200
left  = np.random.normal(0.3, 0.05, N // 2)
right = np.random.normal(0.7, 0.05, N // 2)
consumers = np.clip(np.concatenate([left, right]), 0, 1)

# 2 producers, slightly off-centre
producers_init = np.array([0.4, 0.6])

# Plot the marketplace
fig, ax = plt.subplots(figsize=(9, 2.5))
ax.scatter(consumers, np.zeros_like(consumers), s=20, alpha=0.4, color="#5a5a56", label="consumers (200)")
for i, p in enumerate(producers_init):
    ax.scatter(p, 0.05, s=200, marker="X", color="#993C1D", edgecolor="black", zorder=5,
               label="producers (2)" if i == 0 else None)
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.1, 0.15)
ax.set_yticks([])
ax.set_xlabel("taste axis (e.g. warmth → formality)")
ax.set_title("Initial marketplace — bimodal consumers, two producers near centre")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()


## §3 — Two scoring rules

Given producer positions $P$ and consumer positions $C$, build the utility matrix $U_{ic} = 1 - |P_i - C_c|$. The rule converts $U$ into a single score per producer.

- **Plain winrate.** Each consumer is "won" by the closest producer. Producer $i$'s score = the *fraction of consumers* who chose $i$. Standard leaderboard scoring (LMArena, Chatbot Arena). **Winner-takes-all per consumer.**
- **Weighted winrate.** When producer $i$ wins consumer $c$, $i$ receives credit equal to its *margin* over the runner-up: $U_{ic}^{(\text{top})} - U_{ic}^{(\text{second})}$. Producers are credited more for *dominating* a consumer than for narrowly winning. Donahue & Raghavan's proposed alternative — rewards *comparative advantage*.


In [ ]:
def utility_matrix(producers, consumers):
    """U[i, c] = utility of producer i to consumer c. Closer = higher."""
    return 1 - np.abs(producers[:, None] - consumers[None, :])

def winrate(producers, consumers):
    """Plain winrate: fraction of consumers each producer wins."""
    U = utility_matrix(producers, consumers)
    winners = np.argmax(U, axis=0)
    return np.array([(winners == i).mean() for i in range(len(producers))])

def weighted_winrate(producers, consumers):
    """Weighted winrate: each consumer gives the winner credit = top margin over second."""
    U = utility_matrix(producers, consumers)
    sorted_u = np.sort(U, axis=0)            # sort utilities per consumer
    margins = sorted_u[-1] - sorted_u[-2]    # top minus second per consumer
    winners = np.argmax(U, axis=0)
    scores = np.zeros(len(producers))
    for c_idx, w_idx in enumerate(winners):
        scores[w_idx] += margins[c_idx]
    return scores / len(consumers)

# Quick sanity check
print(f"Initial scores (producers at {producers_init.tolist()}):")
print(f"  plain winrate    = {winrate(producers_init, consumers).round(3)}")
print(f"  weighted winrate = {weighted_winrate(producers_init, consumers).round(3)}")


## §4 — Best-response dynamics

Each producer wants to maximise its score. We simulate strategic behaviour by iterating *best response*:

1. Pick a producer $i$.
2. Holding the other producer fixed, search a grid of positions in $[0, 1]$ for the one maximising $i$'s score.
3. Move $i$ there.
4. Repeat for each producer in turn, for $T$ rounds.

This is a classical best-response dynamic. The equilibrium it converges to **depends on the scoring rule.**


In [ ]:
def best_response(producers, consumers, rule, idx, grid=100):
    """Find the position in [0, 1] maximizing producer idx's score, holding others fixed."""
    best_pos, best_score = producers[idx], -np.inf
    for x in np.linspace(0, 1, grid):
        candidate = producers.copy()
        candidate[idx] = x
        s = rule(candidate, consumers)[idx]
        if s > best_score:
            best_pos, best_score = x, s
    return best_pos

def simulate(init, consumers, rule, T=10):
    """Run T rounds of best-response. Returns the position history [T+1, M]."""
    P = init.copy()
    history = [P.copy()]
    for _ in range(T):
        for i in range(len(P)):
            P[i] = best_response(P, consumers, rule, i)
        history.append(P.copy())
    return np.array(history)

history_winrate  = simulate(producers_init, consumers, winrate)
history_weighted = simulate(producers_init, consumers, weighted_winrate)

print("Final positions after 10 rounds:")
print(f"  plain winrate     → {history_winrate[-1].round(3)}")
print(f"  weighted winrate  → {history_weighted[-1].round(3)}")


## §5 — Where do producers end up?

Side-by-side: trajectory of each producer over the 10 rounds.

- **Plain winrate** — each producer's best response is to crowd next to the other near a population peak. They collapse to **one cluster** and *abandon the other half of consumers.*
- **Weighted winrate** — each producer's best response is to *own* a niche where it dominates. They split cleanly: one producer per consumer cluster.

Same consumers, same starting positions, ten iterations of best response. Only the scoring rule differs.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

for ax, history, title in [
    (axes[0], history_winrate,  "Plain winrate"),
    (axes[1], history_weighted, "Weighted winrate"),
]:
    # Consumers as a backdrop
    ax.scatter(consumers, np.zeros_like(consumers), s=15, alpha=0.3, color="#9a9a94")
    # Producer trajectories: y = round number
    T = len(history)
    for i in range(history.shape[1]):
        ax.plot(history[:, i], np.arange(T), "-", color="#993C1D", alpha=0.4, linewidth=1)
        ax.scatter(history[:, i], np.arange(T), s=30, color="#993C1D", alpha=0.5, zorder=3)
        # Final position bigger
        ax.scatter(history[-1, i], T - 1, s=200, marker="X", color="#6c5ce7",
                   edgecolor="black", zorder=5)
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-1, T)
    ax.set_xlabel("taste axis")
    ax.set_title(title)

axes[0].set_ylabel("round (later → up)")
plt.tight_layout()
plt.show()


## §6 — Consumer welfare

The producer ecosystem is the *means*; consumer welfare is the *end*. For each consumer, what is the best utility they receive under the equilibrium of each rule?


In [ ]:
def best_utility_per_consumer(producers, consumers):
    return utility_matrix(producers, consumers).max(axis=0)

util_winrate  = best_utility_per_consumer(history_winrate[-1],  consumers)
util_weighted = best_utility_per_consumer(history_weighted[-1], consumers)

# CDF
fig, ax = plt.subplots(figsize=(8.5, 4.5))
pct = np.linspace(0, 100, len(consumers))
ax.plot(pct, np.sort(util_winrate),  label=f"plain winrate    (mean = {util_winrate.mean():.3f})",  color="#993C1D")
ax.plot(pct, np.sort(util_weighted), label=f"weighted winrate (mean = {util_weighted.mean():.3f})", color="#0F6E56")
ax.fill_between(pct, np.sort(util_winrate), np.sort(util_weighted),
                where=(np.sort(util_weighted) > np.sort(util_winrate)),
                alpha=0.15, color="#0F6E56", label="welfare gain from weighted winrate")
ax.set_xlabel("consumer percentile (worst-served → best-served)")
ax.set_ylabel("best utility from any producer")
ax.set_title("Consumer welfare CDF under each scoring rule")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

n_worst = len(consumers) // 10
print(f"Worst-served 10% of consumers (n = {n_worst}):")
print(f"  plain winrate    → mean utility = {np.sort(util_winrate)[:n_worst].mean():.3f}")
print(f"  weighted winrate → mean utility = {np.sort(util_weighted)[:n_worst].mean():.3f}")


## §7 — Takeaway

Under **plain winrate**, each producer's best response is to move toward the densest consumer crowd; the rule rewards *winning* consumers regardless of margin, so producers end up clustered. Half the consumer base is left dramatically under-served.

Under **weighted winrate**, each producer's best response is to find a *niche where it dominates*; the rule rewards *margin of victory*, so producers spread out to cover the consumer distribution. The worst-served consumers do markedly better.

Same population. Same producers. Same number of iterations. **Only the scoring rule changes — and it determines whether the model ecosystem looks like a monoculture or a healthy variety.**

This is Donahue & Raghavan's central claim, in miniature: **a leaderboard is an aggregation rule over producers**, and the choice of scoring function is normative, not technical. Plain winrate sacrifices niche-preference welfare; weighted winrate keeps it. There is no neutral leaderboard.

## §8 — Try it yourself

Knobs to turn (re-run the simulation cells with new values):

- **Number of producers `M`.** With 4 producers, plain winrate may still cluster all 4 at one peak; weighted winrate distributes them more evenly. (Change `producers_init` and re-simulate.)
- **Consumer distribution.** Try unimodal (`np.random.normal(0.5, 0.15, N)`), uniform, or skewed. Does plain winrate always cluster at the *mean* of consumers, or at a *mode*?
- **Asymmetric clusters.** Make the left cluster twice as large as the right. Under plain winrate, do both producers chase the bigger cluster?
- **A third rule.** Implement a "minimum utility" rule that scores producers by `min over consumers of utility`. (Maximin → strongly egalitarian.) Does it look more like winrate or weighted winrate in equilibrium?
